# 15.11 Recommender evaluation & bias

Recommender logs are not neutral data; they are traces of what an older policy chose to show.

This notebook closes Part 15 by measuring recall@3 under exposure bias. It contrasts naive logged metrics with inverse propensity scoring, clipping, and position-bias correction.

Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt

RNG = np.random.default_rng(1507)


def softmax(scores, temperature=1.0):
    values = np.asarray(scores, dtype=float) / float(temperature)
    values = values - np.max(values)
    weights = np.exp(values)
    return weights / np.sum(weights)


def normalize_rows(matrix):
    values = np.asarray(matrix, dtype=float)
    norms = np.linalg.norm(values, axis=1, keepdims=True)
    norms = np.maximum(norms, 1e-12)
    return values / norms


def top_k_indices(scores, k):
    values = np.asarray(scores, dtype=float)
    order = np.argsort(-values)
    return order[:k]


def recall_at_k(scores, positives, k=3):
    values = np.asarray(scores, dtype=float)
    hits = []
    for row, pos in zip(values, positives):
        top = set(top_k_indices(row, k))
        wanted = set(np.atleast_1d(pos).astype(int))
        hits.append(len(top & wanted) / max(1, len(wanted)))
    return float(np.mean(hits))


def dcg_at_k(relevance, k=3):
    rel = np.asarray(relevance, dtype=float)[:k]
    gains = np.power(2.0, rel) - 1.0
    discounts = np.log2(np.arange(2, len(rel) + 2))
    return float(np.sum(gains / discounts))


def ndcg_at_k(scores, relevance, k=3):
    values = np.asarray(scores, dtype=float)
    rel = np.asarray(relevance, dtype=float)
    order = np.argsort(-values)[:k]
    ideal = np.argsort(-rel)[:k]
    ideal_dcg = dcg_at_k(rel[ideal], k)
    if ideal_dcg == 0.0:
        return 0.0
    return dcg_at_k(rel[order], k) / ideal_dcg


def mrr_from_scores(scores, target):
    order = np.argsort(-np.asarray(scores, dtype=float))
    rank = int(np.where(order == int(target))[0][0]) + 1
    return 1.0 / rank


def make_f14_ladder(seed=1514):
    rng = np.random.default_rng(seed)
    rungs = []

    item_vectors = np.array([
        [1.0, 0.1, 0.0],
        [0.8, 0.4, 0.0],
        [0.1, 1.0, 0.2],
        [-0.2, 0.3, 0.9],
    ])
    user_vectors = np.array([
        [1.0, 0.2, 0.0],
        [0.2, 1.0, 0.2],
        [-0.1, 0.4, 1.0],
    ])
    relevance = user_vectors @ item_vectors.T
    rungs.append({
        "name": "D1 tiny slate",
        "user_vectors": user_vectors,
        "item_vectors": item_vectors,
        "relevance": relevance,
        "positives": [np.array([0]), np.array([2]), np.array([3])],
        "sessions": [[0, 1, 2], [1, 2, 0], [2, 3, 1]],
        "targets": [2, 0, 1],
        "clicks": np.array([5, 8, 1]),
        "impressions": np.array([100, 100, 20]),
        "eligible": np.array([True, True, True, True]),
        "exposure": np.array([1.0, 0.8, 0.5, 0.3]),
    })

    angles = np.linspace(0.0, 2.0 * np.pi, 8, endpoint=False)
    item_vectors = np.column_stack([np.cos(angles), np.sin(angles), 0.35 * np.cos(2.0 * angles)])
    user_angles = np.array([0.1, 1.6, 3.0, 4.7, 5.6])
    user_vectors = np.column_stack([np.cos(user_angles), np.sin(user_angles), 0.25 * np.sin(user_angles)])
    relevance = user_vectors @ item_vectors.T
    rungs.append({
        "name": "D2 synthetic preferences",
        "user_vectors": user_vectors,
        "item_vectors": item_vectors,
        "relevance": relevance,
        "positives": [np.argsort(-row)[:2] for row in relevance],
        "sessions": [[0, 1, 2, 3], [2, 3, 4], [4, 5, 6], [6, 7, 0], [7, 0, 1]],
        "targets": [3, 4, 6, 0, 1],
        "clicks": np.array([12, 20, 18, 6, 4, 3, 9, 7]),
        "impressions": np.array([200, 260, 220, 150, 80, 60, 120, 140]),
        "eligible": np.ones(8, dtype=bool),
        "exposure": np.linspace(1.0, 0.35, 8),
    })

    item_vectors = rng.normal(size=(12, 4))
    item_vectors = normalize_rows(item_vectors)
    user_vectors = rng.normal(size=(7, 4))
    user_vectors = normalize_rows(user_vectors)
    relevance = user_vectors @ item_vectors.T
    relevance = relevance + rng.normal(scale=0.04, size=relevance.shape)
    exposure = rng.uniform(0.15, 1.0, size=12)
    observed = rng.uniform(size=relevance.shape) < exposure[None, :]
    sparse_relevance = np.where(observed, relevance, -0.2)
    rungs.append({
        "name": "D3 sparse noisy logs",
        "user_vectors": user_vectors,
        "item_vectors": item_vectors,
        "relevance": sparse_relevance,
        "true_relevance": relevance,
        "positives": [np.argsort(-row)[:2] for row in relevance],
        "sessions": [[0, 2, 5, 7], [1, 2, 4], [3, 8, 9], [10, 2, 0], [6, 6, 11], [4, 5, 6], [9, 1, 3]],
        "targets": [7, 4, 9, 0, 11, 6, 3],
        "clicks": np.array([30, 16, 9, 4, 3, 2, 8, 7, 5, 4, 2, 1]),
        "impressions": np.array([600, 280, 190, 90, 45, 30, 120, 110, 70, 65, 22, 12]),
        "eligible": rng.uniform(size=12) > 0.10,
        "exposure": exposure,
    })

    genres = np.array([
        [1, 0, 0, 0, 1],
        [1, 1, 0, 0, 0],
        [0, 1, 1, 0, 0],
        [0, 0, 1, 1, 0],
        [0, 0, 0, 1, 1],
        [1, 0, 0, 1, 0],
        [0, 1, 0, 0, 1],
        [0, 0, 1, 0, 1],
        [1, 0, 1, 0, 0],
        [0, 1, 0, 1, 0],
    ], dtype=float)
    item_vectors = normalize_rows(genres + 0.05 * rng.normal(size=genres.shape))
    user_vectors = normalize_rows(np.array([
        [2, 0, 0, 0, 1],
        [1, 2, 0, 0, 0],
        [0, 1, 2, 0, 0],
        [0, 0, 1, 2, 0],
        [0, 0, 0, 1, 2],
        [1, 0, 1, 0, 1],
    ], dtype=float))
    relevance = user_vectors @ item_vectors.T
    rungs.append({
        "name": "D4 MovieLens-like inline",
        "user_vectors": user_vectors,
        "item_vectors": item_vectors,
        "relevance": relevance,
        "positives": [np.argsort(-row)[:3] for row in relevance],
        "sessions": [[0, 5, 8], [1, 2, 6], [2, 3, 8], [3, 4, 9], [4, 7, 0], [7, 8, 6]],
        "targets": [8, 6, 3, 9, 4, 6],
        "clicks": np.array([50, 34, 25, 18, 17, 14, 9, 7, 6, 5]),
        "impressions": np.array([1000, 700, 520, 430, 320, 250, 150, 110, 90, 80]),
        "eligible": np.ones(10, dtype=bool),
        "exposure": np.array([1.0, 0.9, 0.75, 0.65, 0.55, 0.45, 0.35, 0.30, 0.25, 0.20]),
    })

    head = normalize_rows(rng.normal(loc=0.6, scale=0.25, size=(5, 5)))
    torso = normalize_rows(rng.normal(loc=0.0, scale=0.7, size=(8, 5)))
    cold = normalize_rows(np.array([
        [1, 1, 0, 0, 1],
        [0, 1, 1, 1, 0],
        [1, 0, 1, 0, 1],
        [0, 0, 1, 1, 1],
    ], dtype=float))
    item_vectors = np.vstack([head, torso, cold])
    user_vectors = normalize_rows(rng.normal(size=(8, 5)))
    user_vectors[0] = normalize_rows(np.array([[1, 1, 0, 0, 1]], dtype=float))[0]
    relevance = user_vectors @ item_vectors.T
    popularity = np.array([2000, 1500, 1200, 900, 700, 250, 180, 130, 100, 75, 60, 45, 35, 10, 8, 5, 3])
    clicks = np.maximum(1, np.round(popularity * np.clip(0.03 + 0.04 * np.maximum(item_vectors[:, 0], 0.0), 0.01, 0.20))).astype(int)
    exposure = np.clip(popularity / popularity.max(), 0.02, 1.0)
    rungs.append({
        "name": "D5 long-tail cold-start",
        "user_vectors": user_vectors,
        "item_vectors": item_vectors,
        "relevance": relevance,
        "positives": [np.argsort(-row)[:3] for row in relevance],
        "sessions": [[0, 1, 14], [1, 6, 15], [2, 8, 16], [3, 9, 13], [4, 10, 14], [5, 11, 15], [6, 12, 16], [7, 13, 14]],
        "targets": [14, 15, 16, 13, 14, 15, 16, 14],
        "clicks": clicks,
        "impressions": popularity.astype(int),
        "eligible": np.array([True] * 13 + [True, True, False, True]),
        "exposure": exposure,
        "cold_items": np.array([13, 14, 15, 16]),
    })

    return rungs


## The concept, built once: evaluate biased logs

For logged clicks $c_i$ and propensities $p_i=P(\text{shown}_i)$, inverse propensity scoring estimates

$$\hat V_{IPS}=\frac{1}{n}\sum_i \frac{c_i}{p_i}.$$

The reusable evaluator computes recall@k, exposure-masked positives, IPS with optional clipping, and position-bias correction.

In [ ]:

def biased_eval(scores, true_relevance, exposure, clicks=None, propensities=None, k=3, clip=None, examination=None):
    scores = np.asarray(scores, dtype=float)
    true_relevance = np.asarray(true_relevance, dtype=float)
    exposure = np.asarray(exposure, dtype=float)
    positives = [np.flatnonzero(row > 0.0) for row in true_relevance]
    recall = recall_at_k(scores, positives, k=k)
    observed_relevance = true_relevance * exposure[None, :]
    observed_positives = int(np.sum((true_relevance[0] > 0.0) & (exposure > 0.0)))
    result = {
        "recall": recall,
        "observed_relevance": observed_relevance,
        "observed_positives": observed_positives,
    }
    if clicks is not None and propensities is not None:
        weights = 1.0 / np.maximum(np.asarray(propensities, dtype=float), 1e-12)
        if clip is not None:
            weights = np.minimum(weights, float(clip))
        result["ips"] = float(np.mean(np.asarray(clicks, dtype=float) * weights))
        result["naive"] = float(np.mean(clicks))
    if examination is not None:
        result["position_corrected"] = np.asarray(clicks, dtype=float) / np.maximum(np.asarray(examination, dtype=float), 1e-12)
    return result


## Verify the lesson numbers once

A three-user holdout with every hidden positive in the top 3 has mean recall@3 $1.000$. True relevance $(1,1,0,1)$ with exposure $(1,0,1,0)$ reveals only one of three positives, a 1-of-3 observed-positive warning. For clicks $(1,0,1)$ and propensities $(0.5,0.2,0.1)$, IPS is $4.000$ and naive click mean is $0.667$. Equal relevance $0.5$ under examination $(1.0,0.6,0.3,0.15)$ yields observed rates $(0.500,0.300,0.150,0.075)$.

In [ ]:

scores = np.array([[3.0, 2.0, 1.0, 0.0], [0.2, 3.0, 1.0, 0.0], [0.1, 0.0, 3.0, 2.0]])
positives = [np.array([0, 1]), np.array([1]), np.array([2])]
recall = recall_at_k(scores, positives, k=3)
assert round(recall, 3) == 1.000

true_rel = np.array([1, 1, 0, 1])
exposure = np.array([1, 0, 1, 0])
observed = true_rel * exposure
assert int(np.sum(observed)) == 1
assert int(np.sum(true_rel)) == 3

clicks = np.array([1, 0, 1])
propensities = np.array([0.5, 0.2, 0.1])
ips = float(np.mean(clicks / propensities))
naive = float(np.mean(clicks))
assert round(ips, 3) == 4.000
assert round(naive, 3) == 0.667

position_rates = 0.5 * np.array([1.0, 0.6, 0.3, 0.15])
assert [round(float(x), 3) for x in position_rates] == [0.500, 0.300, 0.150, 0.075]
print("recall", recall, "observed positives", int(np.sum(observed)), "of", int(np.sum(true_rel)), "IPS", ips)


## The dataset ladder: D1 to D5

The same F14 recommender ladder is built inline: tiny slate, synthetic preferences, sparse noisy logs, a MovieLens-like genre matrix, and a long-tail cold-start catalog. The single metric here is **recall@3**.

In [ ]:

rungs = make_f14_ladder()
for rung in rungs:
    user_shape = rung["user_vectors"].shape
    item_shape = rung["item_vectors"].shape
    rel_shape = rung["relevance"].shape
    sparsity = float(np.mean(rung.get("exposure", np.ones(item_shape[0])) < 0.5))
    print(rung["name"], "users", user_shape, "items", item_shape, "relevance", rel_shape, "low exposure fraction", round(sparsity, 3))
    print("sample relevance row", np.round(rung["relevance"][0, : min(5, item_shape[0])], 3))


## Run the same method across D1-D5

Each rung compares true recall@3 with an exposure-biased estimate. D5 makes the feedback-loop problem visible because long-tail exposure is much smaller than head exposure.

In [ ]:

results = []
for rung in rungs:
    scores = rung["relevance"]
    true_relevance = (rung.get("true_relevance", rung["relevance"]) > np.percentile(rung.get("true_relevance", rung["relevance"]), 70)).astype(float)
    exposure = rung["exposure"]
    output = biased_eval(scores, true_relevance, exposure, k=3)
    observed_scores = scores * exposure[None, :]
    observed_metric = recall_at_k(observed_scores, [np.flatnonzero(row > 0.0) for row in true_relevance], k=3)
    results.append({"name": rung["name"], "metric": output["recall"], "observed_metric": observed_metric, "output": output})

for row in results:
    print(f"{row['name']:<28} true_recall@3={row['metric']:.3f} logged_recall@3={row['observed_metric']:.3f}")


## Results visualization

Panels show true versus logged relevance for the first user in each rung, and the curve contrasts true recall@3 with the biased logged estimate.

In [ ]:

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
flat_axes = axes.ravel()
for ax, rung, row in zip(flat_axes[:5], rungs, results):
    true_row = (rung.get("true_relevance", rung["relevance"])[0] > np.percentile(rung.get("true_relevance", rung["relevance"]), 70)).astype(float)
    logged_row = true_row * rung["exposure"]
    width = 0.35
    x = np.arange(min(8, len(true_row)))
    ax.bar(x - width / 2.0, true_row[: len(x)], width=width, label="true")
    ax.bar(x + width / 2.0, logged_row[: len(x)], width=width, label="logged")
    ax.set_title(rung["name"])
    ax.set_xlabel("item")
    ax.set_ylabel("relevance")
flat_axes[0].legend(fontsize=8)
flat_axes[5].plot(range(1, 6), [row["metric"] for row in results], marker="o", label="true")
flat_axes[5].plot(range(1, 6), [row["observed_metric"] for row in results], marker="s", label="logged")
flat_axes[5].set_xticks(range(1, 6))
flat_axes[5].set_xticklabels(["D1", "D2", "D3", "D4", "D5"])
flat_axes[5].set_ylim(0.0, 1.05)
flat_axes[5].set_title("recall@3 curve")
flat_axes[5].legend()
fig.tight_layout()


## Pitfall on D5: treating logs as random samples

A head-heavy old policy can make a model look better offline simply because it resembles the logging policy. IPS helps when propensities are known, but small propensities require clipping and variance checks.

In [ ]:

d5 = rungs[-1]
clicks = (d5["relevance"][0, :6] > np.median(d5["relevance"][0])).astype(float)
propensities = d5["exposure"][:6]
unclipped = biased_eval(d5["relevance"][:1], (d5["relevance"][:1] > 0.0).astype(float), d5["exposure"], clicks=clicks, propensities=propensities, clip=None)
clipped = biased_eval(d5["relevance"][:1], (d5["relevance"][:1] > 0.0).astype(float), d5["exposure"], clicks=clicks, propensities=propensities, clip=5.0)
exam = np.array([1.0, 0.6, 0.3, 0.15, 0.1, 0.08])
position_corrected = clicks / exam
print("naive", round(unclipped["naive"], 3))
print("unclipped IPS", round(unclipped["ips"], 3))
print("clipped IPS", round(clipped["ips"], 3))
print("position-corrected clicks", np.round(position_corrected, 3))


## Evaluate it + practice

- Report **recall@3 with biased and corrected estimates** against a no-skill popularity or random baseline on every rung.
- Sanity check that the D1 worked numbers match the lesson asserts before trusting larger rungs.
- Ablation: evaluate only exposed head items and ignore propensities; the metric should drop on D3-D5.
- Failure signals: head-item collapse, cold items never appearing, or a metric curve that improves only because labels are biased.
- Keep splits chronological or exposure-aware when the lesson's pitfall involves time or logging.

Practice: change the cutoff from 3 to 5 and compare the metric curve.

Practice: vary the clipping threshold and discuss bias-variance tradeoffs.